In [8]:
import json
from pathlib import Path
PROJECT_ROOT = Path().resolve().parents[0] 
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
original_test_path = RAW_DATA_DIR / "UCF_Crimes-Train-Test-Split" / "Anomaly_Detection_splits" / "Anomaly_Test.txt"
uca_test_path = RAW_DATA_DIR / "UCF_Annotation" / "json" / "UCFCrime_Test.json"
print(f"Checking original TXT at: {original_test_path.exists()}")
print(f"Checking UCA JSON at: {uca_test_path.exists()}")

Checking original TXT at: True
Checking UCA JSON at: True


In [9]:
original_videos = set()

with open(original_test_path, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if parts:
            # Use .stem instead of .name to remove the .mp4 extension
            video_name = Path(parts[0]).stem 
            original_videos.add(video_name)

print(f"Total videos in original Anomaly_Test.txt: {len(original_videos)}")

Total videos in original Anomaly_Test.txt: 290


In [10]:
uca_videos = set()

with open(uca_test_path, 'r') as f:
    uca_data = json.load(f)

raw_uca_video_paths = []
if isinstance(uca_data, dict):
    raw_uca_video_paths = list(uca_data.keys())
elif isinstance(uca_data, list):
    first_item = uca_data[0]
    for key in ['video_name', 'video_id', 'video_path', 'name']:
        if key in first_item:
            raw_uca_video_paths = [item[key] for item in uca_data]
            break

for v in raw_uca_video_paths:
    # Use .stem here as well to ensure a perfectly clean comparison
    uca_videos.add(Path(v).stem)

print(f"Total videos in UCA UCFCrime_Test.json: {len(uca_videos)}")

Total videos in UCA UCFCrime_Test.json: 310


In [11]:
missing_in_uca = original_videos - uca_videos
extra_in_uca = uca_videos - original_videos

print("--- INTEGRITY CHECK RESULTS ---")
if not missing_in_uca and not extra_in_uca:
    print("✅ PERFECT MATCH!")
    print("The UCA test set contains the exact same videos as the original dataset.")
    print("You are cleared to use the UCA JSONs for valid SOTA benchmarking.")
else:
    print("❌ MISMATCH DETECTED!")
    
    if missing_in_uca:
        print(f"\nVideos in Original but missing in UCA ({len(missing_in_uca)}):")
        for v in sorted(missing_in_uca)[:10]: # Print first 10 for inspection
            print(f"  - {v}")
        if len(missing_in_uca) > 10: print("  - ...")
            
    if extra_in_uca:
        print(f"\nVideos in UCA but not in Original ({len(extra_in_uca)}):")
        for v in sorted(extra_in_uca)[:10]:
            print(f"  - {v}")
        if len(extra_in_uca) > 10: print("  - ...")

--- INTEGRITY CHECK RESULTS ---
❌ MISMATCH DETECTED!

Videos in Original but missing in UCA (240):
  - Abuse028_x264
  - Abuse030_x264
  - Arrest001_x264
  - Arrest007_x264
  - Arrest024_x264
  - Arrest030_x264
  - Arrest039_x264
  - Arson007_x264
  - Arson009_x264
  - Arson010_x264
  - ...

Videos in UCA but not in Original (260):
  - Abuse037_x264
  - Abuse038_x264
  - Abuse039_x264
  - Abuse040_x264
  - Abuse041_x264
  - Abuse042_x264
  - Arrest043_x264
  - Arrest044_x264
  - Arrest046_x264
  - Arrest048_x264
  - ...


In [12]:
# Paths for UCA Train and Val
uca_train_path = RAW_DATA_DIR / "UCF_Annotation" / "json" / "UCFCrime_Train.json"
uca_val_path = RAW_DATA_DIR / "UCF_Annotation" / "json" / "UCFCrime_Val.json"

def get_videos_from_uca_json(json_path):
    videos = set()
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    if isinstance(data, dict):
        raw_paths = list(data.keys())
    elif isinstance(data, list):
        first = data[0]
        for key in ['video_name', 'video_id', 'video_path', 'name']:
            if key in first:
                raw_paths = [item[key] for item in data]
                break
                
    for v in raw_paths:
        videos.add(Path(v).stem)
    return videos

uca_train_videos = get_videos_from_uca_json(uca_train_path)
uca_val_videos = get_videos_from_uca_json(uca_val_path)

# Check intersections (Data Leakage)
train_leakage = original_videos.intersection(uca_train_videos)
val_leakage = original_videos.intersection(uca_val_videos)

print("--- DATA LEAKAGE CHECK ---")
if not train_leakage and not val_leakage:
    print("✅ NO LEAKAGE! The UCA Train/Val sets are completely isolated from the Original Test set.")
    print("We can safely use UCA for training/validation, and the Original for testing.")
else:
    print("❌ FATAL DATA LEAKAGE DETECTED!")
    print(f"Original Test videos found in UCA Train: {len(train_leakage)}")
    print(f"Original Test videos found in UCA Val: {len(val_leakage)}")

--- DATA LEAKAGE CHECK ---
❌ FATAL DATA LEAKAGE DETECTED!
Original Test videos found in UCA Train: 214
Original Test videos found in UCA Val: 22
